In [ ]:
import re
import numpy as np
import random
from sympy import parse_expr, count_ops, Function
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm
import os
from torch.utils.data import DataLoader


os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"


full_dataset = load_dataset("gsm8k", "main", split="train")
random_indices = random.sample(range(len(full_dataset)), 3000)
dataset = full_dataset.select(random_indices)
print(f"Selected 3000 random samples from {len(full_dataset)} total problems")


tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-128k-instruct")
model = AutoModelForCausalLM.from_pretrained("microsoft/phi-3-mini-128k-instruct", 
                                           torch_dtype=torch.float16, device_map="auto")


def generate_cot_paths_batch(questions, num_paths=3):
    all_paths = [[] for _ in range(len(questions))]
    
    for path_idx in range(num_paths):
        prompts = [f"Solve this problem step by step. Question: {q}\nAnswer:" for q in questions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, 
                                   max_new_tokens=256,
                                   output_scores=True,
                                   return_dict_in_generate=True)
        

        transition_scores = model.compute_transition_scores(
            outputs.sequences, outputs.scores, normalize_logits=True
        )
        

        for i in range(len(questions)):

            sample_scores = transition_scores[i].cpu().numpy().flatten().tolist()
            

            decoded = tokenizer.decode(outputs.sequences[i])
            backtracks = len(re.findall(r'(actually|correction|but|however)', decoded))
            
            all_paths[i].append({
                'steps': decoded.split('\n'),
                'probs': sample_scores,
                'backtracks': backtracks
            })
    
    return all_paths


def analyze_complexity(answer):
    try:

        equation = re.search(r'\b\d+[\s\S]*=\s*\d+', answer)
        if equation is None:

            operations = len(re.findall(r'[\+\-\*\/\(\)]', answer))
            return {'variables': 0, 'functions': 0, 'depth': operations}
            
        equation = equation.group()
        expr = parse_expr(equation.split('=')[0])
        
        return {
            'variables': len(expr.free_symbols),
            'functions': len(expr.atoms(Function)),
            'depth': count_ops(expr, visual=False)
        }
    except:

        operations = len(re.findall(r'[\+\-\*\/\(\)]', answer))
        return {'variables': 0, 'functions': 0, 'depth': operations}


def calculate_metrics_batch(questions, answers, batch_size=8):

    all_scores = []
    
    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i:i+batch_size]
        batch_answers = answers[i:i+batch_size]
        batch_scores = []
        
        try:

            batch_paths = generate_cot_paths_batch(batch_questions)
            
            for j, (question, answer, paths) in enumerate(zip(batch_questions, batch_answers, batch_paths)):
                try:

                    step_variances = []
                    for path in paths:
                        step_probs = path['probs']
                        if len(step_probs) > 0:
                            step_variances.append(np.var(step_probs))
                    confidence_div = np.mean(step_variances) if step_variances else 0
                    
                 
                    backtracks = np.mean([p['backtracks'] for p in paths])
                    
                  
                    complexity = analyze_complexity(answer)
                    sc_score = min((complexity['variables']*0.4 + 
                                   complexity['functions']*0.3 + 
                                   complexity['depth']*0.3), 1.0)
                    
                    batch_scores.append(0.4*confidence_div + 0.3*backtracks + 0.3*sc_score)
                except Exception as e:
                    print(f"Error processing individual question: {e}")
                    batch_scores.append(0)
            
            all_scores.extend(batch_scores)
        except Exception as e:
            print(f"Error processing batch: {e}")
            all_scores.extend([0] * len(batch_questions))
    
    return all_scores


def process_dataset_in_batches(dataset, batch_size=8):
  
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    scored_problems = []
    current_idx = 0 
    
    for batch in tqdm(dataloader, desc="Scoring problems in batches"):
        questions = batch['question']
        answers = batch['answer']
        
      
        scores = calculate_metrics_batch(questions, answers, batch_size)
        
       
        for i in range(len(questions)):
            scored_problems.append({
                'question': questions[i],
                'answer': answers[i],
                'score': scores[i],
                'original_index': random_indices[current_idx + i] 
            })
        
        current_idx += len(questions)
    
    return scored_problems


batch_size = 8 
scored_problems = process_dataset_in_batches(dataset, batch_size)


scored_problems.sort(key=lambda x: x['score'], reverse=True)
subset_size = int(len(scored_problems) * 0.2)
selected_subset = scored_problems[:subset_size]
selected_indices = [item['original_index'] for item in selected_subset] 
print(f"Selected {subset_size} problems (20% of 3000) based on difficulty score")


import json
with open("gsm8k_selected_subset_indices.json", "w") as f:
    json.dump({
        "metadata": {
            "original_dataset_size": len(full_dataset),
            "random_sample_size": len(dataset),
            "selected_subset_size": len(selected_subset),
            "selection_method": "0.4*ConfidenceDivergence + 0.3*BacktrackFrequency + 0.3*SymbolicComplexity"
        },
        "selected_indices": selected_indices
    }, f, indent=2)

print("Selected subset indices saved to gsm8k_selected_subset_indices.json")


validation_subset = full_dataset.select(selected_indices)
print(f"Subset accuracy: {validate_subset_batch(model, validation_subset):.2%}")



import json
with open("gsm8k_selected_subset.json", "w") as f:
    json.dump({
        "metadata": {
            "original_dataset_size": len(full_dataset),
            "random_sample_size": len(dataset),
            "selected_subset_size": len(selected_subset),
            "selection_method": "0.4*ConfidenceDivergence + 0.3*BacktrackFrequency + 0.3*SymbolicComplexity"
        },
        "selected_subset": [
            {"question": item["question"], "answer": item["answer"], "score": item["score"]} 
            for item in selected_subset
        ]
    }, f, indent=2)

print("Selected subset saved to gsm8k_selected_subset.json")


Selected 3000 random samples from 7473 total problems


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Scoring problems in batches: 100%|██████████| 375/375 [2:34:44<00:00, 24.76s/it]  

Selected 600 problems (20% of 3000) based on difficulty score
Selected subset indices saved to gsm8k_selected_subset_indices.json


NameError: name 'validate_subset_batch' is not defined

In [ ]:
import re
import numpy as np
import random
from sympy import parse_expr, count_ops, Function
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm


full_dataset = load_dataset("gsm8k", "main", split="train")
random_indices = random.sample(range(len(full_dataset)), 3000)
dataset = full_dataset.select(random_indices)
print(f"Selected 3000 random samples from {len(full_dataset)} total problems")


tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-128k-instruct")
model = AutoModelForCausalLM.from_pretrained("microsoft/phi-3-mini-128k-instruct", 
                                           torch_dtype=torch.float16, device_map="auto")


def generate_cot_paths(question, num_paths=3):
    paths = []
    prompt = f"Solve this problem step by step. Question: {question}\nAnswer:"
    
    for _ in range(num_paths):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, 
                               max_new_tokens=256,
                               output_scores=True,
                               return_dict_in_generate=True)
        

        transition_scores = model.compute_transition_scores(
            outputs.sequences, outputs.scores, normalize_logits=True
        )
        probs = torch.exp(transition_scores).cpu().numpy().flatten().tolist()
        

        decoded = tokenizer.decode(outputs.sequences[0])
        backtracks = len(re.findall(r'(actually|correction|but|however)', decoded))
        
        paths.append({
            'steps': decoded.split('\n'),
            'probs': probs,
            'backtracks': backtracks
        })
    
    return paths


def analyze_complexity(answer):
    try:
  
        equation = re.search(r'\b\d+[\s\S]*=\s*\d+', answer)
        if equation is None:
         
            operations = len(re.findall(r'[\+\-\*\/\(\)]', answer))
            return {'variables': 0, 'functions': 0, 'depth': operations}
            
        equation = equation.group()
        expr = parse_expr(equation.split('=')[0])
        
        return {
            'variables': len(expr.free_symbols),
            'functions': len(expr.atoms(Function)),
            'depth': count_ops(expr, visual=False)
        }
    except:

        operations = len(re.findall(r'[\+\-\*\/\(\)]', answer))
        return {'variables': 0, 'functions': 0, 'depth': operations}


def calculate_metrics(question, answer):
    try:
        cot_paths = generate_cot_paths(question)
        

        step_variances = []
        for path in cot_paths:
            step_probs = path['probs']
            step_variances.append(np.var(step_probs))
        confidence_div = np.mean(step_variances)
        
   
        backtracks = np.mean([p['backtracks'] for p in cot_paths])
        
   
        complexity = analyze_complexity(answer)
        sc_score = min((complexity['variables']*0.4 + 
                       complexity['functions']*0.3 + 
                       complexity['depth']*0.3), 1.0)
        
        return 0.4*confidence_div + 0.3*backtracks + 0.3*sc_score
    
    except Exception as e:
        print(f"Error processing question: {e}")
        return 0


scored_problems = []
for item in tqdm(dataset, desc="Scoring problems"):
    score = calculate_metrics(item['question'], item['answer'])
    scored_problems.append({
        'question': item['question'],
        'answer': item['answer'],
        'score': score
    })


scored_problems.sort(key=lambda x: x['score'], reverse=True)
subset_size = int(len(scored_problems) * 0.2)
selected_subset = scored_problems[:subset_size]
print(f"Selected {subset_size} problems (20% of 3000) based on difficulty score")


def validate_subset(model, subset):
    correct = 0
    for problem in tqdm(subset, desc="Validating"):
        inputs = tokenizer(problem['question'], return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256)
        answer = tokenizer.decode(outputs[0])
        
 
        pred = re.findall(r'\d+\.?\d*', answer.split("####")[-1] if "####" in answer else answer)
        gold = re.findall(r'\d+\.?\d*', problem['answer'].split("####")[-1])
        
        if pred and gold and pred[-1] == gold[-1]:
            correct += 1
            
    return correct/len(subset)

print(f"Subset accuracy: {validate_subset(model, selected_subset):.2%}")


import json
with open("gsm8k_selected_subset.json", "w") as f:
    json.dump({
        "metadata": {
            "original_dataset_size": len(full_dataset),
            "random_sample_size": len(dataset),
            "selected_subset_size": len(selected_subset),
            "selection_method": "0.4*ConfidenceDivergence + 0.3*BacktrackFrequency + 0.3*SymbolicComplexity"
        },
        "selected_subset": [
            {"question": item["question"], "answer": item["answer"], "score": item["score"]} 
            for item in selected_subset
        ]
    }, f, indent=2)

print("Selected subset saved to gsm8k_selected_subset.json")
